In [1]:
import pandas as pd
import numpy as np
import os
from coniferest.pineforest import PineForest
from coniferest.session import Session
from coniferest.session.callback import (TerminateAfter, prompt_decision_callback,)
from coniferest.session.callback import (TerminateAfter, viewer_decision_callback,)
from coniferest.label import Label

In [2]:
from tqdm import tqdm

def load_single(oid_filename, feature_filename):
    oid     = np.memmap(oid_filename, mode='c', dtype=np.uint64)
    feature = np.memmap(feature_filename, mode='c', dtype=np.float32).reshape(oid.shape[0], -1)
    return oid, feature

In [3]:
fields = [686, 769]

In [4]:
all_oids = []
all_features = []

for field in fields:
    oid_filename = f"/media2/SNAD/dr23-features-collected_by_field/dr23_oid_{field}.dat"
    feature_filename = f"/media2/SNAD/dr23-features-collected_by_field/dr23_feature_{field}.dat"

    oids_field, features_field = load_single(oid_filename, feature_filename)

    all_oids.append(np.array(oids_field))
    all_features.append(np.array(features_field))

oids_use = np.concatenate(all_oids)
features_use = np.concatenate(all_features)
oids_use


array([686201100000000, 686201100000001, 686201100000002, ...,
       769216400076366, 769216400076368, 769216400076369],
      shape=(13635400,), dtype=uint64)

In [5]:
data2 = pd.read_csv("/media/tomy/git/canidates_log - Sheet5.csv", header = None)
data2.columns = ["url"]
data_art = data2["url"].str.split("/").str[-1]
    

In [6]:
data = pd.read_parquet('/media/tomy/git/top_5000_iter_006.parquet')
pd.set_option('display.max_rows', None)
data_2500 = data.iloc[0:2500]
OIDs = []

for o, inp in enumerate (data_2500['id'], start=0):
    oid_part, mjd_part = inp.split('_')
    oid = int(oid_part.removeprefix('ZTFDR'))
    OIDs.append((o, oid))

In [7]:
fields = ["686", "769"]

In [8]:
labels_coniferest = []
g_field = []

for i in OIDs:
    idx, oid = i
    field = str(oid)[:3]
    if field in fields :
        g_field.append(oid)
        if str(oid) in data_art.values:
            labels_coniferest.append(Label.REGULAR)
        else:
            labels_coniferest.append(Label.ANOMALY)

#for i in labels_coniferest :
    #print (i)

#print (len(labels_coniferest))
#for h in g_field :             
   #print (h)

In [9]:
final_features = []
final_labels = []
final_oids = []

oids_use_str = oids_use.astype(str)

for oid, label in zip(g_field, labels_coniferest):
    oid_str = str(oid)
    match = np.where(oids_use_str == oid_str)[0]
    if len(match) > 0:
        idx = match[0]
        final_features.append(features_use[idx])
        final_labels.append(label)
        final_oids.append(oid)

final_features = np.array(final_features)
final_labels = np.array(final_labels)
final_oids = np.array(final_oids)

In [12]:
model_ztf_p = PineForest(random_seed=42)

model_ztf.fit_known(
    features_use,
    known_data=final_features,
    known_labels=final_labels
)

In [13]:
data = features_use
metadata = oids_use

In [14]:
model_ztf = PineForest(42)

class RecordCallback:
    def __init__(self):
        self.records = []

    def __call__(self, metadata, data, session):
        decision = session.last_decision
        
        if decision == 1:
            label = "REGULAR"
        elif decision == -1:
            label = "ANOMALY"
        else:
            label = "UNKNOWN"
        self.records.append(f'{metadata} -> {label}')

    def print_report(self):
        print('Records:')
        print('\n'.join(self.records))
        
record_callback = RecordCallback()

session_ztf = Session(
    data=data,
    metadata=metadata,
    model=model_ztf_p,
    # Prompt for a decision and open object's page on the SNAD Viewer
    decision_callback=viewer_decision_callback,
    on_decision_callbacks=[
        record_callback,
        TerminateAfter(100),
    ]
)
session_ztf.run()
record_callback.print_report()

Check https://ztf.snad.space/view/686212100039979 for details
Is 686212100039979 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Check https://ztf.snad.space/view/686210200109516 for details
Is 686210200109516 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Check https://ztf.snad.space/view/769206200102995 for details
Is 769206200102995 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

...

Check https://ztf.snad.space/view/686213200143319 for details
Is 686213200143319 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Check https://ztf.snad.space/view/686204200060506 for details
Is 686204200060506 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Records:
686212100039979 -> REGULAR
686210200109516 -> REGULAR
769206200102995 -> REGULAR
769211200029836 -> REGULAR
769208400049447 -> REGULAR
686210100013764 -> REGULAR
686205400085022 -> REGULAR
686201100008938 -> REGULAR
686216400043210 -> REGULAR
686216300008813 -> REGULAR
769208200048227 -> REGULAR
686204100120026 -> REGULAR
686210300143390 -> REGULAR
769208300130769 -> REGULAR
769215100026909 -> REGULAR
769209200081067 -> REGULAR
686214400028823 -> REGULAR
686206300153304 -> REGULAR
686209300077954 -> REGULAR
686206200094015 -> REGULAR
686215200064926 -> REGULAR
686211100023686 -> REGULAR
686205200029885 -> REGULAR
769201200057408 -> REGULAR
686213100060388 -> REGULAR
686213300040010 -> REGULAR
769205300083690 -> REGULAR
686201400018926 -> REGULAR
769209100077774 -> REGULAR
769208300102988 -> REGULAR
769204300023814 -> REGULAR
769207100039473 -> REGULAR
769201400036113 -> REGULAR
686210200110148 -> REGULAR
686203300196030 -> REGULAR
769201400038351 -> REGULAR
686214300044102 -> 

In [15]:
df_records = pd.DataFrame(record_callback.records,columns=["Report"])
df_records.to_csv("/media/tomy/git/report_callback_priors2.csv", index=False)